In [4]:
import mediapipe as mp
import cv2
import playsound
from threading import Thread
import time
import numpy as np

In [3]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

ALARM = "../alarm-clock-90867.mp3"


In [ ]:
# # 1. Cabeça projetada para frente (principal indicador)
# def Cabeca_projetada_para_frente(landmarks):

#     nose = landmarks[mp_holistic.PoseLandmark.NOSE]
#     left_Shouder = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER]
#     right_Shouder = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]
     
#     posturaPadrao= 0.55
#     nose_z = nose.z
#     shoulder_z = (left_Shouder.z + right_Shouder.z) / 2
#     # ajustando o limite de acordo com a largura dos ombros, adaptável a diferentes pessoas e distancias da câmera
#     larguraOmbros = abs(left_Shouder.x - right_Shouder.x)
#     limite = larguraOmbros * 0.5
#     # limite = 0.09

#     diff_atual = shoulder_z - nose_z
#     cabeca_projetada_para_frente = False
#     if diff_atual > posturaPadrao + limite : 
#         cabeca_projetada_para_frente = True
      
#     return { "diffAtaul":  f'{diff_atual:.2f}',"posturaP + limite":f'{posturaPadrao+limite:.2f}', "cabecaProjetada":cabeca_projetada_para_frente}

In [6]:
def sound_alarm(path=ALARM):
    playsound.playsound(path)
    

# 1. Cabeça projetada para frente (principal indicador)
def desvioZ_suavizado(desvioZ, history, window=30):
    history.append(desvioZ)
    if len(history) > window:
        history.pop(0)
    return np.mean(history)

def Cabeca_projetada_para_frente(landmarks):
    nose     = landmarks[mp_holistic.PoseLandmark.NOSE]
    left_sh  = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER]
    right_sh = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]

    shoulder_mid_z = (left_sh.z + right_sh.z) / 2.0

    # para divisao por 0 não acontecer
    if abs(shoulder_mid_z) < 0.001:
        return None

    # abs(-5 )= 5, abs(-0.0001)= 0.0001 
    diff = (shoulder_mid_z - nose.z) / abs(shoulder_mid_z) # normalizar a diferença para funcionar independente da distância da câmera.

    return diff


# 2. Inclinação da cabeça (pescoço torto)
def Inclinacao_cabeca(landmarks):
    left_ear = landmarks[mp_holistic.PoseLandmark.LEFT_EAR]
    right_ear = landmarks[mp_holistic.PoseLandmark.RIGHT_EAR]
    
    DiferencaEarAltura = abs(left_ear.y - right_ear.y)
    inclinada= False
    if DiferencaEarAltura > 0.05:  
        inclinada = True
    return {"diferenca": DiferencaEarAltura,  "inclinacao": inclinada}




In [12]:
FORWARD_HISTORY = []
FORWARD_HEAD_THRESH = 1.78

cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic: 
    # poderia ser usado uma variavel (holistic.close() necessario), mas é usado assim devido liberação de memória autoática ...
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Make Detections
        results = holistic.process(image)
        
        if results.pose_landmarks is None:
            print("No pose landmarks detected.")
        else:
            landmarks = results.pose_landmarks.landmark

     # 1. Cabeça projetada para frente (principal indicador)
            desvioZ = Cabeca_projetada_para_frente(landmarks)
            if desvioZ is not None:
                desvioZ_suave = desvioZ_suavizado(desvioZ, FORWARD_HISTORY)
                print(desvioZ_suave)
                if desvioZ_suave < FORWARD_HEAD_THRESH:
                    print('True', desvioZ_suave)
                else:
                    print('False', desvioZ_suave)

                    
    #  2. Inclinação da cabeça (pescoço torto)
            # cabecaInclinada = Inclinacao_cabeca(landmarks)
            
            # print(cabecaInclinada["inclinacao"])
  
 




            # Recolor image back to BGR for rendering
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
          
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                    mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2)
                                    )

        # mostrando na tela
        cv2.namedWindow('Raw Webcam Feed', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Raw Webcam Feed', 1100, 800)
        cv2.imshow('Raw Webcam Feed', cv2.flip(image,1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
            
cap.release()
cv2.destroyAllWindows()


2.779303668788369
False 2.779303668788369
2.7368595780276683
False 2.7368595780276683
2.7885907824941203
False 2.7885907824941203
2.8058388901738582
False 2.8058388901738582
2.7431720230884564
False 2.7431720230884564
2.703174414355908
False 2.703174414355908
2.669558554893268
False 2.669558554893268
2.598170057741293
False 2.598170057741293
2.539615438027406
False 2.539615438027406
2.5003971296852865
False 2.5003971296852865
2.4792034652364836
False 2.4792034652364836
2.4595658525569326
False 2.4595658525569326
2.44074698230622
False 2.44074698230622
2.4302643499417784
False 2.4302643499417784
2.422517880539978
False 2.422517880539978
2.4161927729362556
False 2.4161927729362556
2.408446632535998
False 2.408446632535998
2.399133056939581
False 2.399133056939581
2.3948941924787084
False 2.3948941924787084
2.39175054928209
False 2.39175054928209
2.386623997485588
False 2.386623997485588
2.382406372107437
False 2.382406372107437
2.370848844556654
False 2.370848844556654
2.3587817794310086

In [8]:
# fechar camera/ janelas
cap.release()
cv2.destroyAllWindows()

In [13]:
# calcular baseline


def calcular_PosturaPAdrao(landmarks,valores):
    # duração da coleta (em segundos)

    nose = landmarks[mp_holistic.PoseLandmark.NOSE]
    left = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER]
    right = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]

    # centro dos ombros
    shoulder_z = (left.z + right.z) / 2

    # diferença
    diff = shoulder_z - nose.z
    print("Diff:", diff)
    valores.append(diff)


    
CALIBRATION_TIME = 10
valores = []
start_time = time.time()

cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic: 
    # poderia ser usado uma variavel (holistic.close() necessario), mas é usado assim devido liberação de memória autoática ...
    
    while cap.isOpened() and  time.time() - start_time < CALIBRATION_TIME:
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Make Detections
        results = holistic.process(image)
        
        if results.pose_landmarks is None:
            print("No pose landmarks detected.")
        else:
            landmarks = results.pose_landmarks.landmark

     # 1. Cabeça projetada para frente (principal indicador)
            calcular_PosturaPAdrao(landmarks,valores)
            
            # Recolor image back to BGR for rendering
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            # # 4. Pose Detections
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                    mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2)
                                    )

        # mostrando na tela
        cv2.namedWindow('Raw Webcam Feed', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Raw Webcam Feed', 1100, 800)
        cv2.imshow('Raw Webcam Feed', cv2.flip(image,1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    baseline = sum(valores) / len(valores)
    print("---------------concluída! -------------------")
    print(f'minVal: {min(valores)}, maxVal: {max(valores)}')
    print("PosturaPadrao:", baseline)
    print("Amostras:", len(valores))
            
cap.release()
cv2.destroyAllWindows()


Diff: 0.5731858164072037
Diff: 0.6124804615974426
Diff: 0.6098033338785172
Diff: 0.5911265313625336
Diff: 0.5693816542625427
Diff: 0.5726070553064346
Diff: 0.5618864893913269
Diff: 0.5683750063180923
Diff: 0.5924280881881714
Diff: 0.5984546989202499
Diff: 0.6017596125602722
Diff: 0.6003267914056778
Diff: 0.5778238773345947
Diff: 0.5759661942720413
Diff: 0.5762020200490952
Diff: 0.581340566277504
Diff: 0.5667983144521713
Diff: 0.5594232678413391
Diff: 0.6448822468519211
Diff: 0.6245114356279373
Diff: 0.6185207515954971
Diff: 0.5981354415416718
Diff: 0.6537632495164871
Diff: 0.6639291346073151
Diff: 0.6513147801160812
Diff: 0.6523348838090897
Diff: 0.6788584291934967
Diff: 0.6830447018146515
Diff: 0.674845740199089
Diff: 0.6652622371912003
Diff: 0.6467970162630081
Diff: 0.6272918283939362
Diff: 0.6214254051446915
Diff: 0.6227296739816666
Diff: 0.6465961784124374
Diff: 0.5922277271747589
Diff: 0.5690062344074249
Diff: 0.5712501257658005
Diff: 0.5741302967071533
Diff: 0.5803069472312927
Di